### Insurance Premium Prediction Model
Getting the lifestyle details from an user and predicting if they are a high or low paying customer

In [112]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score

In [113]:
df=pd.read_csv('C:\\Niladri\\empty\\coding\\FastAPI\\datasets\\insurance_premium_prediction_balanced_dataset.csv')

In [114]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
397,73,65.4,1.86,4.98821,False,Visakhapatnam,unemployed,Low
1328,44,84.3,1.69,8.66801,False,Gurgaon,government_job,Low
183,31,65.4,1.72,6.28974,False,Indore,private_job,Low
241,23,81.0,1.69,8.87186,False,Kolkata,self_employed,Low
541,49,63.6,1.62,8.68586,True,Pune,self_employed,High


In [115]:
df['occupation'].unique()

array(['government_job', 'business_owner', 'retired', 'self_employed',
       'private_job', 'student', 'unemployed'], dtype=object)

In [116]:
df_feat=df.copy()

### FEATURE ENGINEERING

In [117]:
#feature1: BMI
df_feat['bmi'] = round(df_feat['weight'] / (df_feat['height'] ** 2), 2)

In [118]:
#feature2: Age Group
def age_group(age):
    if age < 18:
        return 'Child'
    elif 18 <= age < 30:
        return 'Young Adult'
    elif 30 <= age < 50:
        return 'Adult'
    else:
        return 'Senior'

df_feat['age_group'] = df_feat['age'].apply(age_group)

In [119]:
#feature3: Lifestyle Risk
def lifestyle_risk(row):
    smoker = row['smoker']
    BMI = row['bmi']
    if smoker and BMI >= 30:
        return 'High'
    elif smoker or BMI >= 30:
        return 'Medium'
    else:
        return 'Low'

df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk, axis=1) #Training and API feature engineering must be identical.

In [120]:
tier1_cities=["Mumbai","Delhi","Bangalore","Hyderabad","Ahmedabad","Chennai","Kolkata","Pune","Jaipur","Surat"]
tier2_cities=["Lucknow","Kanpur","Nagpur","Indore","Thane","Bhopal","Visakhapatnam","Pimpri-Chinchwad","Patna","Vadodara"]

In [121]:
#feature4: City Tier
def city_tier(city):
    if city in tier1_cities:
        return 'Tier 1'
    elif city in tier2_cities:
        return 'Tier 2'
    else:
        return 'Tier 3'

df_feat['city_tier'] = df_feat['city'].apply(city_tier)

In [122]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city' ])

,income_lpa,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
0,10.12698,government_job,Medium,26.88,Senior,Low,Tier 2
1,11.09231,business_owner,Low,20.28,Young Adult,Low,Tier 1
2,8.78471,retired,Medium,18.86,Adult,Low,Tier 2
3,9.03622,self_employed,Medium,32.95,Adult,Medium,Tier 3
4,14.60966,self_employed,Medium,20.32,Senior,Low,Tier 1
...,...,...,...,...,...,...,...
1500,13.24743,self_employed,Medium,19.01,Senior,Low,Tier 2
1501,13.73834,government_job,Low,21.77,Adult,Low,Tier 1
1502,6.87936,government_job,High,39.37,Young Adult,Medium,Tier 1
1503,7.13010,retired,Low,23.17,Senior,Low,Tier 2


In [123]:
x=df_feat.drop(columns=['insurance_premium_category'])
y=df_feat['insurance_premium_category']

In [124]:
categorical_features = ['age_group', 'lifestyle_risk', 'occupation', 'city_tier']
numerical_features = ['bmi','income_lpa']

In [125]:
#create column transformer for ohe
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

In [126]:
#create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [127]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1, stratify=y)
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [128]:
y_pred = pipeline.predict(X_test)
# accuracy = accuracy_score(y_test, y_pred)
# accuracy

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Precision
print("Precision:", precision_score(
    y_test, y_pred, average='weighted'
))

# Recall
print("Recall:", recall_score(
    y_test, y_pred, average='weighted'
))

# F1 Score
print("F1 Score:", f1_score(
    y_test, y_pred, average='weighted'
))

# Detailed report for each class
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7475083056478405
Precision: 0.7526556458682517
Recall: 0.7475083056478405
F1 Score: 0.749065022020101

Classification Report:
              precision    recall  f1-score   support

        High       0.81      0.70      0.75        61
         Low       0.83      0.83      0.83       150
      Medium       0.59      0.63      0.61        90

    accuracy                           0.75       301
   macro avg       0.74      0.72      0.73       301
weighted avg       0.75      0.75      0.75       301


Confusion Matrix:
[[ 43   3  15]
 [  0 125  25]
 [ 10  23  57]]


In [129]:
import pickle
# Save the trained model to a file
pickle_model_path="model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)